# Sleep Apnea Multimodal — Experiment 3 (preprocessing ablation) on Colab

Five arms, each changing exactly one preprocessing decision, on window 30 s.

| arm | what changes | source cache |
|---|---|---|
| `baseline` | nothing (this is the current pipeline) | filtered |
| `no_clip` | amplitude clipping removed | filtered |
| `clip_per_record` | clip bound scaled to each record's own std | filtered |
| `norm_per_record` | z-score per record instead of per dataset | filtered |
| `no_filter` | no band-pass / low-pass / detrend | unfiltered |

**Caveat that must travel with `no_filter`:** the unfiltered cache is still resampled
with `resample_poly`, and that anti-alias step is itself a low-pass (ECG ~50 Hz,
respiration ~12.5 Hz). The arm means "no added filter", not "no filtering at all".

This notebook expects `research_cache.zip` to contain both `artifacts/signals/` and
`artifacts/signals_unfiltered/`. Run `python scripts/21_make_cache_zip.py` locally to
rebuild it (about 800 MB).

## 0. GPU check

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU.')
print('device:', torch.cuda.get_device_name(0))

## 1. Mount Drive and extract both caches

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil, pathlib

ZIP = '/content/drive/MyDrive/research/research_cache.zip'
ROOT = pathlib.Path('/content/Research_v2')

assert os.path.exists(ZIP), f'not found: {ZIP}'
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(ROOT)
os.chdir(ROOT)
print('extracted to', ROOT)
for d in ('artifacts/signals', 'artifacts/signals_unfiltered'):
    p = ROOT / d
    n = len(list(p.glob('*.npy'))) if p.exists() else 0
    print(f'{d}: {n} .npy files')

## 2. Verify both caches align

The unfiltered cache must have identical sample counts to the filtered one, otherwise
window start times would point at the wrong samples. This check aborts if they differ.

In [ ]:
import numpy as np, pathlib
main = pathlib.Path('artifacts/signals'); unf = pathlib.Path('artifacts/signals_unfiltered')
bad = []
for f in sorted(main.glob('*_ecg100.npy')):
    g = unf / f.name
    a = np.load(f, mmap_mode='r'); b = np.load(g, mmap_mode='r')
    if a.shape != b.shape:
        bad.append((f.name, a.shape, b.shape))
print('records checked:', len(list(main.glob('*_ecg100.npy'))))
print('shape mismatches:', len(bad))
assert not bad, bad[:5]

## 3. Smoke test one arm before spending GPU time

Small cap and one epoch. Confirms all five arms load and train. Takes a couple of
minutes; a bug here would otherwise cost an hour. `--val-only` avoids reading test
during the check, so the test split still stays untouched for the real run.

In [ ]:
!rm -rf /content/Research_v2/artifacts/ablation/smoke
!python scripts/23_run_ablation.py --seeds 42 --models resp_only --duration 30 \
    --device cuda --workers 2 --max-epochs 1 --train-cap 512 --val-only \
    --out artifacts/ablation/smoke

## 4. Full run: 5 arms x 2 models x 3 seeds, window 30 s

About 80 minutes on a T4. If the session dies, rerun cells 1-2 and start this cell
again: finished runs have `resume_<model>.pt` and skip to the next epoch.

Start with one seed if you want a cheaper read first:
`--seeds 42` takes about a third of the time.

In [ ]:
!python scripts/23_run_ablation.py --seeds 42,1337,2024 \
    --models proposed,resp_only --duration 30 --device cuda --workers 2

## 5. Copy results back to Drive

In [ ]:
DEST = '/content/drive/MyDrive/research/results'
os.makedirs(DEST, exist_ok=True)
src = ROOT / 'artifacts' / 'ablation'
if src.exists():
    shutil.copytree(src, pathlib.Path(DEST) / 'ablation', dirs_exist_ok=True)
    print('copied ablation results')
os.system(f'find {DEST}/ablation -maxdepth 2 -name "*.csv" | head -20')

## 6. After it finishes

1. Download `results/ablation/` from Drive into the local repo at
   `artifacts/ablation/seed42-1337-2024_30s/`.
2. Record in `LOG_KERJA.md` that Experiment 3 ran on a T4 GPU (seed 2024 of Experiment 2
   already did), because floating-point order differs from CPU runs.
3. Report the `no_filter` caveat: the anti-alias resampling remains, so the arm tests
*added* filtering, not all filtering.